# Notebook 02 - Locked publication training: Agent B (ViT-B/16)

This notebook trains Agent B only from the locked `train.csv` and `model_val.csv` publication artifacts. Checkpoint selection and early stopping use `model_val` exclusively. Full runs write the best model and reproducibility metadata under `artifacts/rescue/checkpoints`; `ARGUS_SMOKE_TEST=1` redirects every output to a temporary directory.


## 1. Environment setup and publication-safety helpers

Install the training dependencies, locate the Argus modules, configure locked split/checkpoint
roots, and define hash, fingerprint, overlap, smoke-test, and run-metadata helpers.

In [ ]:
import sys, os
from pathlib import Path


def _find_ml_training():
    for root in (".", "..", "/kaggle/working", "/kaggle/input"):
        if not os.path.isdir(root):
            continue
        for dirpath, _dirs, files in os.walk(root):
            if "config.py" in files and "transforms.py" in files:
                if dirpath not in sys.path:
                    sys.path.insert(0, dirpath)
                return Path(dirpath).resolve()
    raise FileNotFoundError("Could not locate the Argus ml_training directory.")


ML_TRAINING_DIR = _find_ml_training()
REPO_ROOT = ML_TRAINING_DIR.parent
print("Argus ml_training discovered at:", ML_TRAINING_DIR)

from transforms import get_train_transform, get_eval_transform
from weighting import effective_number_weights
from config import EFFECTIVE_NUMBER_BETA

import sys, subprocess
print("NOTE: Kaggle Internet must be ON for package installation and full-run pretrained weights.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "timm", "grad-cam", "torchmetrics"], check=True)

import glob, random, math, warnings, json, hashlib, tempfile, platform
import importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
import timm
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, roc_auc_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SMOKE_TEST = os.environ.get("ARGUS_SMOKE_TEST", "0") == "1"
os.environ.setdefault("ARGUS_RUN_VISUALIZATIONS", "0")
RUN_VISUALIZATIONS = os.environ["ARGUS_RUN_VISUALIZATIONS"] == "1"
DEFAULT_ARTIFACT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else REPO_ROOT
ARTIFACT_ROOT = Path(os.environ.get("ARGUS_ARTIFACT_ROOT", str(DEFAULT_ARTIFACT_ROOT))).resolve()
SPLIT_DIR = Path(os.environ.get("ARGUS_SPLIT_DIR", str(REPO_ROOT / "artifacts" / "rescue" / "splits"))).resolve()
PUBLICATION_CHECKPOINT_DIR = ARTIFACT_ROOT / "artifacts" / "rescue" / "checkpoints"

if SMOKE_TEST:
    CHECKPOINT_DIR = Path(tempfile.mkdtemp(prefix="argus_publication_smoke_"))
    print("SMOKE TEST: checkpoints and metadata redirected to", CHECKPOINT_DIR)
else:
    CHECKPOINT_DIR = PUBLICATION_CHECKPOINT_DIR
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESUME_DIR = CHECKPOINT_DIR / "resume"
RESUME_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR = CHECKPOINT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

ISIC_CLASSES = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMAGE_SIZE = 224
NUM_CLASSES = 8


def _sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _normalized_image_id(value):
    return Path(str(value).strip()).stem.lower()


def _class_counts(frame):
    return {str(k): int(v) for k, v in frame["label"].value_counts().sort_index().items()}


def _assert_no_forbidden_split_path_variables(namespace):
    blocked = ("risk" + "_dev", "final" + "_test")
    offenders = []

    def visit(name, value):
        if isinstance(value, Path):
            value = str(value)
        if isinstance(value, str):
            low = value.lower()
            if any(token in low for token in blocked) and (".csv" in low or "\\" in low or "/" in low):
                offenders.append(name)
        elif isinstance(value, dict):
            for key, item in value.items():
                visit(f"{name}[{key!r}]", item)
        elif isinstance(value, (list, tuple, set)):
            for index, item in enumerate(value):
                visit(f"{name}[{index}]", item)

    for name, value in namespace.items():
        if not name.startswith("__"):
            visit(name, value)
    assert not offenders, "Forbidden held-out split path found in notebook variables: " + ", ".join(offenders)


def _load_locked_training_splits(split_dir):
    train_path = split_dir / "train.csv"
    model_val_path = split_dir / "model_val.csv"
    summary_path = split_dir / "split_summary.json"
    fingerprint_path = split_dir / "dataset_fingerprint.json"
    for path in (train_path, model_val_path, summary_path, fingerprint_path):
        if not path.is_file():
            raise FileNotFoundError(f"Required locked publication artifact is missing: {path}")

    allowed_csv_names = {train_path.name, model_val_path.name}

    def keep_allowed_manifest_pairs(pairs):
        return {
            key: value for key, value in pairs
            if not (isinstance(key, str) and key.lower().endswith(".csv") and key not in allowed_csv_names)
        }

    with summary_path.open("r", encoding="utf-8") as handle:
        raw_summary = json.load(handle, object_pairs_hook=keep_allowed_manifest_pairs)
    with fingerprint_path.open("r", encoding="utf-8") as handle:
        fingerprint = json.load(handle)

    allowed = {"train": train_path, "model_val": model_val_path}
    allowed_hashes = {}
    allowed_summary = {}
    for split_name, csv_path in allowed.items():
        expected = raw_summary.get("csv_sha256", {}).get(csv_path.name)
        actual = _sha256_file(csv_path)
        assert expected and actual == expected, (
            f"Locked split hash mismatch for {csv_path.name}: expected={expected}, actual={actual}"
        )
        allowed_hashes[csv_path.name] = actual
        allowed_summary[split_name] = raw_summary["splits"][split_name]

    frames = {name: pd.read_csv(path) for name, path in allowed.items()}
    required = {"image", "label", "lesion_id", "lesion_group", "research_split"}
    for split_name, frame in frames.items():
        missing = sorted(required - set(frame.columns))
        assert not missing, f"{split_name}.csv missing required columns: {missing}"
        values = set(frame["research_split"].dropna().astype(str))
        assert values == {split_name}, f"{split_name}.csv has invalid research_split values: {sorted(values)}"
        expected_info = allowed_summary[split_name]
        assert len(frame) == int(expected_info["rows"]), f"{split_name}.csv row count disagrees with manifest"
        assert _class_counts(frame) == {str(k): int(v) for k, v in expected_info["class_counts"].items()}, (
            f"{split_name}.csv class counts disagree with manifest"
        )

    train_images = set(frames["train"]["image"].map(_normalized_image_id))
    val_images = set(frames["model_val"]["image"].map(_normalized_image_id))
    assert not (train_images & val_images), "train/model_val image overlap detected"

    def present(values):
        return {str(v).strip() for v in values if pd.notna(v) and str(v).strip()}

    train_lesions = present(frames["train"]["lesion_id"])
    val_lesions = present(frames["model_val"]["lesion_id"])
    assert not (train_lesions & val_lesions), "train/model_val lesion_id overlap detected"
    train_groups = present(frames["train"]["lesion_group"])
    val_groups = present(frames["model_val"]["lesion_group"])
    assert not (train_groups & val_groups), "train/model_val lesion_group overlap detected"

    assert fingerprint.get("split_protocol") == "publication_rescue_v1", "Unexpected dataset fingerprint protocol"
    assert fingerprint.get("image_col") == "image" and fingerprint.get("label_col") == "label", (
        "Dataset fingerprint column contract does not match the training notebooks"
    )
    assert fingerprint.get("lesion_id_col") == "lesion_id", "Dataset fingerprint lesion column mismatch"
    assert int(fingerprint["n_rows"]) == int(raw_summary["total_rows"]), (
        "Dataset fingerprint row count disagrees with split manifest"
    )
    aggregate_counts = {}
    for info in raw_summary["splits"].values():
        for label, count in info["class_counts"].items():
            aggregate_counts[str(label)] = aggregate_counts.get(str(label), 0) + int(count)
    assert aggregate_counts == {str(k): int(v) for k, v in fingerprint["label_counts"].items()}, (
        "Dataset fingerprint label counts disagree with split manifest"
    )
    assert float(fingerprint["lesion_id_coverage_pct"]) >= 95.0, "Lesion metadata coverage is below 95%"
    image_digest = str(fingerprint.get("image_id_sha256", ""))
    assert len(image_digest) == 64 and all(c in "0123456789abcdef" for c in image_digest.lower()), (
        "Dataset fingerprint image-id digest is invalid"
    )

    safe_manifest = {
        "total_rows": int(raw_summary["total_rows"]),
        "splits": allowed_summary,
        "csv_sha256": allowed_hashes,
    }
    return frames["train"], frames["model_val"], safe_manifest, fingerprint, _sha256_file(fingerprint_path)


def _discover_image_dir(root="/kaggle/input"):
    override = os.environ.get("ARGUS_IMAGE_DIR")
    if override:
        path = Path(override).resolve()
        assert path.is_dir(), f"ARGUS_IMAGE_DIR does not exist: {path}"
        return path
    best_dir, best_count = None, 0
    for dirpath, _dirs, files in os.walk(root):
        count = sum(
            name.lower().startswith("isic_") and name.lower().endswith((".jpg", ".jpeg"))
            for name in files
        )
        if count > best_count:
            best_dir, best_count = Path(dirpath), count
    assert best_dir is not None, f"Could not find ISIC image files under {root}"
    print("Discovered image directory:", best_dir, f"({best_count:,} images)")
    return best_dir


def _attach_image_paths(frame, image_dir):
    image_index = {}
    for path in image_dir.iterdir():
        if path.is_file() and path.suffix.lower() in (".jpg", ".jpeg", ".png"):
            image_index[_normalized_image_id(path.name)] = str(path)
    result = frame.copy()
    result["path"] = result["image"].map(lambda value: image_index.get(_normalized_image_id(value)))
    missing = int(result["path"].isna().sum())
    assert missing == 0, f"{missing} locked split image(s) are missing from {image_dir}"
    return result


def _tiny_class_subset(frame, rows_per_class):
    subset = frame.groupby("label", group_keys=False, sort=True).head(rows_per_class).copy()
    return subset.sort_values(["label", "image"], kind="mergesort").reset_index(drop=True)


def _git_commit():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=str(REPO_ROOT), text=True, stderr=subprocess.DEVNULL
        ).strip()
    except Exception:
        if SMOKE_TEST:
            return "unavailable-in-smoke-test"
        raise RuntimeError("Publication training requires a Git checkout so the run commit can be recorded.")


def _package_versions():
    packages = ["torch", "torchvision", "timm", "numpy", "pandas", "scikit-learn", "Pillow", "torchmetrics"]
    versions = {"python": platform.python_version()}
    for package in packages:
        try:
            versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            versions[package] = "not-installed"
    return versions


BEST_SELECTION = {}


def _selection_key(path):
    return str(Path(path).resolve())


def _record_selection(path, epoch, stage, metrics):
    record = {"epoch": int(epoch), "stage": stage, "validation_metrics": metrics}
    BEST_SELECTION[_selection_key(path)] = record
    sidecar = RESUME_DIR / (Path(path).stem + "_selection.json")
    sidecar.write_text(json.dumps(record, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def _restore_selection(path):
    key = _selection_key(path)
    if key in BEST_SELECTION:
        return BEST_SELECTION[key]
    sidecar = RESUME_DIR / (Path(path).stem + "_selection.json")
    if sidecar.is_file():
        BEST_SELECTION[key] = json.loads(sidecar.read_text(encoding="utf-8"))
        return BEST_SELECTION[key]
    return None


def _write_run_metadata(agent_id, checkpoint_path, architecture, pretrained_source,
                        optimizer_config, scheduler_config, loss_config, sampling_config,
                        validation_metrics, transform_config):
    selection = _restore_selection(checkpoint_path)
    assert selection is not None and selection.get("epoch") is not None, (
        "Cannot publish run metadata without the checkpoint-selection epoch"
    )
    payload = {
        "agent": agent_id,
        "smoke_test": SMOKE_TEST,
        "git_commit": _git_commit(),
        "split_manifest_hashes": SPLIT_MANIFEST["csv_sha256"],
        "dataset_fingerprint": DATASET_FINGERPRINT,
        "dataset_fingerprint_file_sha256": DATASET_FINGERPRINT_SHA256,
        "model_architecture": architecture,
        "pretrained_checkpoint_source": pretrained_source,
        "pretrained_enabled_for_this_run": PRETRAINED_ENABLED,
        "random_seed": SEED,
        "optimizer": optimizer_config,
        "scheduler": scheduler_config,
        "loss": loss_config,
        "sampling_method": sampling_config,
        "epoch_selected": int(selection["epoch"]),
        "selection_stage": selection["stage"],
        "validation_metrics": validation_metrics,
        "checkpoint_sha256": _sha256_file(checkpoint_path),
        "checkpoint_path": str(Path(checkpoint_path)),
        "transform_configuration": transform_config,
        "package_versions": _package_versions(),
    }
    metadata_path = CHECKPOINT_DIR / f"{agent_id}_run_metadata.json"
    metadata_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    print("Run metadata:", metadata_path)
    print("Checkpoint SHA-256:", payload["checkpoint_sha256"])
    return metadata_path


FULL_NAMES = {
    "MEL": "Melanoma", "NV": "Melanocytic nevus", "BCC": "Basal cell carcinoma",
    "AK": "Actinic keratosis", "BKL": "Benign keratosis", "DF": "Dermatofibroma",
    "VASC": "Vascular lesion", "SCC": "Squamous cell carcinoma",
}
BATCH_SIZE = 8 if SMOKE_TEST else 32
NUM_WORKERS = 0 if SMOKE_TEST else 2
MODEL_NAME = "vit_base_patch16_224.augreg_in21k_ft_in1k"
PRETRAINED_SOURCE = "timm vit_base_patch16_224.augreg_in21k_ft_in1k (ImageNet-21k -> ImageNet-1k)"
PRETRAINED_ENABLED = not SMOKE_TEST
CHECKPOINT_PATH = str(CHECKPOINT_DIR / "agent_b_best.pth")
WORK_DIR = str(CHECKPOINT_DIR)


In [ ]:
# ===== LOCKED PUBLICATION RUN CONFIG =====
import config as _cfg
print("=" * 60)
print("ARGUS LOCKED PUBLICATION RUN")
print(f"  smoke_test={SMOKE_TEST} | device={DEVICE} | seed={SEED}")
print(f"  IMAGE_SIZE={_cfg.IMAGE_SIZE} | TRAINING_MODE={_cfg.TRAINING_MODE!r} | BETA={_cfg.EFFECTIVE_NUMBER_BETA}")
print(f"  STAGE_A/B epochs={_cfg.STAGE_A_EPOCHS}/{_cfg.STAGE_B_EPOCHS} | joint max={_cfg.PHASE2_MAX_EPOCHS} | patience={_cfg.PHASE2_PATIENCE}")
print(f"  checkpoint={CHECKPOINT_PATH}")
print(f"  resume_dir={RESUME_DIR}")
print("=" * 60)


## 2. Locked train/model-validation data

Loads and verifies only the two development artifacts authorized for model fitting. Hashes, research roles, fingerprint aggregates, and image/lesion disjointness are checked before any image is opened.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

sns.set_theme(style="whitegrid")

TRAIN_CSV = SPLIT_DIR / "train.csv"
MODEL_VAL_CSV = SPLIT_DIR / "model_val.csv"
train_df, model_val_df, SPLIT_MANIFEST, DATASET_FINGERPRINT, DATASET_FINGERPRINT_SHA256 = (
    _load_locked_training_splits(SPLIT_DIR)
)
_assert_no_forbidden_split_path_variables(globals())

IMAGE_DIR = _discover_image_dir()
train_df = _attach_image_paths(train_df, IMAGE_DIR)
model_val_df = _attach_image_paths(model_val_df, IMAGE_DIR)
if SMOKE_TEST:
    train_df = _tiny_class_subset(train_df, 1)
    model_val_df = _tiny_class_subset(model_val_df, 2)
    print(f"SMOKE TEST subset: train={len(train_df)} model_val={len(model_val_df)}")

df = pd.concat([train_df, model_val_df], ignore_index=True)
labels = df["label"].to_numpy(dtype=np.int64)
print(f"Locked train={len(train_df):,} | locked model_val={len(model_val_df):,}")
print("Allowed CSV hashes:", SPLIT_MANIFEST["csv_sha256"])
print("Dataset fingerprint SHA-256:", DATASET_FINGERPRINT_SHA256)


In [ ]:
# Class-distribution bar chart over the eight ISIC classes.
counts = df["label"].value_counts().sort_index()
class_counts = pd.Series(
    [int(counts.get(idx, 0)) for idx in range(NUM_CLASSES)],
    index=ISIC_CLASSES,
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=class_counts.index, y=class_counts.values, ax=ax,
            hue=class_counts.index, palette="mako", legend=False)
ax.set_title("ISIC-8 class distribution (locked development data)")
ax.set_xlabel("Class")
ax.set_ylabel("Number of images")
for patch, value in zip(ax.patches, class_counts.values):
    ax.annotate(f"{int(value):,}",
                (patch.get_x() + patch.get_width() / 2.0, patch.get_height()),
                ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

print("Per-class counts:")
for name, value in class_counts.items():
    print(f"  {name:5s} ({FULL_NAMES[name]:25s}): {int(value):,}")

In [ ]:
# 3x3 grid: one representative sample image per class (wraps if <9 classes).
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(3, 3, figsize=(11, 11))
for ax_idx, ax in enumerate(axes.ravel()):
    class_idx = ax_idx % NUM_CLASSES
    subset = df[df["label"] == class_idx]
    if len(subset) == 0:
        ax.axis("off")
        continue
    row = subset.iloc[int(rng.integers(0, len(subset)))]
    img = Image.open(row["path"]).convert("RGB")
    ax.imshow(img)
    ax.set_title(f"{ISIC_CLASSES[class_idx]} - {FULL_NAMES[ISIC_CLASSES[class_idx]]}", fontsize=10)
    ax.axis("off")
plt.suptitle("ISIC-8 sample dermoscopic images", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Dataset, samplers, and DataLoaders

The canonical train/evaluation transforms are applied to the locked research roles. `model_val` is the only source used for early stopping and checkpoint selection.


In [ ]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler


class ISICDataset(Dataset):
    """Reads locked publication rows and returns (image_tensor, label_int, image_path)."""
    def __init__(self, frame, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        path = row["path"]
        image = Image.open(path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, int(row["label"]), path


def build_train_transforms(size=IMAGE_SIZE):
    return get_train_transform()


def build_val_transforms(size=IMAGE_SIZE):
    return get_eval_transform()


train_dataset = ISICDataset(train_df, transform=build_train_transforms())
val_dataset = ISICDataset(model_val_df, transform=build_val_transforms())


In [ ]:
# Effective-number-of-samples class weights (Cui et al.), normalized to mean 1.
train_labels = train_df["label"].to_numpy()
class_sample_counts = np.bincount(train_labels, minlength=NUM_CLASSES).astype(np.float64)
class_sample_counts = np.clip(class_sample_counts, 1.0, None)

class_weights = effective_number_weights(class_sample_counts, beta=EFFECTIVE_NUMBER_BETA)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
print("Class weights (inv-sqrt, mean 1):",
      {name: round(float(w), 3) for name, w in zip(ISIC_CLASSES, class_weights)})

# Per-sample weights drive the WeightedRandomSampler to balance minibatches.
sample_weights = class_weights[train_labels]
sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)

pin = (DEVICE == "cuda")
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=pin, drop_last=not SMOKE_TEST)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=pin)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")


## 4. Model: ViT-B/16 with a frozen backbone

Create an ImageNet-21k -> 1k fine-tuned ViT-B/16 (`vit_base_patch16_224.augreg_in21k_ft_in1k`)
with an 8-class head via `timm`. Freeze the backbone so Phase 1 trains only the head.

In [ ]:
import timm

# timm ViT-B/16; the -U install above ensures this model id resolves.
model = timm.create_model(MODEL_NAME, pretrained=PRETRAINED_ENABLED, num_classes=NUM_CLASSES)
model = model.to(DEVICE)

# Freeze everything, then unfreeze only the classifier head for Phase 1.
for p in model.parameters():
    p.requires_grad = False
for p in model.get_classifier().parameters():
    p.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params (head only): {trainable:,} / {total:,}")
print("Num transformer blocks:", len(model.blocks))


In [ ]:
from sklearn.metrics import balanced_accuracy_score
from torchmetrics.classification import MulticlassAUROC


class FocalLoss(nn.Module):
    """Multi-class focal loss with optional per-class alpha weighting.

    loss = (1 - p_t) ** gamma * CE, where p_t is the unweighted true-class probability.
    """

    def __init__(self, gamma=2.0, alpha=None, reduction="mean"):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction
        if alpha is not None:
            self.register_buffer("alpha", alpha)
        else:
            self.alpha = None

    def forward(self, logits, targets):
        log_probs = F.log_softmax(logits, dim=1)
        log_pt = log_probs.gather(dim=1, index=targets.unsqueeze(1)).squeeze(1)
        pt = log_pt.exp()
        base_ce = -log_pt
        if self.alpha is not None:
            alpha_t = self.alpha.to(device=logits.device, dtype=logits.dtype)[targets]
            base_ce = base_ce * alpha_t
        loss = (1.0 - pt) ** self.gamma * base_ce
        if self.reduction == "mean":
            return loss.mean()
        if self.reduction == "sum":
            return loss.sum()
        return loss


auroc_metric = MulticlassAUROC(num_classes=NUM_CLASSES, average="macro").to(DEVICE)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))


@torch.no_grad()
def evaluate(network, loader):
    """Return (balanced_accuracy, macro_auc, y_true, y_pred)."""
    network.eval()
    auroc_metric.reset()
    all_preds, all_targets = [], []
    for images, targets, _paths in loader:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
            probs = F.softmax(network(images), dim=1)
        auroc_metric.update(probs, targets)
        all_preds.append(probs.argmax(dim=1).detach().cpu().numpy())
        all_targets.append(targets.detach().cpu().numpy())
    preds_arr = np.concatenate(all_preds)
    targets_arr = np.concatenate(all_targets)
    bal_acc = float(balanced_accuracy_score(targets_arr, preds_arr))
    macro_auc = float(auroc_metric.compute().item())
    return bal_acc, macro_auc, targets_arr, preds_arr


def run_epoch(network, loader, criterion, optimizer):
    """Run one mixed-precision optimization epoch; return mean train loss."""
    network.train()
    running_loss, num_batches = 0.0, 0
    for images, targets, _paths in loader:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
            logits = network(images)
            loss = criterion(logits, targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += float(loss.item())
        num_batches += 1
    return running_loss / max(num_batches, 1)


history = {"loss": [], "bal_acc": [], "macro_auc": []}
best_macro_auc = -1.0

## 5. Phase 1 - head-only training

Train only the classifier head with `FocalLoss(alpha=None, gamma=2.0)` and
`AdamW(lr=1e-3)`. Each epoch logs validation balanced accuracy + macro one-vs-rest AUC
and snapshots the best checkpoint to `artifacts/rescue/checkpoints/agent_b_best.pth`.

In [ ]:
MAX_EPOCHS_HEAD = 1 if SMOKE_TEST else 5
MAX_EPOCHS_FINETUNE = 15
WEIGHT_DECAY = 1e-4
LR_HEAD = 1e-3
BASE_LR = 1e-5            # Phase 2 base LR
ATTN_LR_SCALE = 0.1      # attention-layer params train at base_lr * 0.1
FOCAL_GAMMA = 2.0

criterion = FocalLoss(gamma=FOCAL_GAMMA, alpha=None).to(DEVICE)

head_optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR_HEAD, weight_decay=WEIGHT_DECAY,
)

for epoch in range(1, MAX_EPOCHS_HEAD + 1):
    train_loss = run_epoch(model, train_loader, criterion, head_optimizer)
    bal_acc, macro_auc, _, _ = evaluate(model, val_loader)
    history["loss"].append(train_loss)
    history["bal_acc"].append(bal_acc)
    history["macro_auc"].append(macro_auc)
    print(f"[P1 {epoch:02d}/{MAX_EPOCHS_HEAD}] "
          f"loss={train_loss:.4f} bal_acc={bal_acc:.4f} macro_auc={macro_auc:.4f}")
    if not math.isnan(macro_auc) and macro_auc > best_macro_auc:
        best_macro_auc = macro_auc
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        _record_selection(CHECKPOINT_PATH, epoch, "head", {"balanced_accuracy": float(bal_acc), "macro_auc": float(macro_auc)})
        print(f"    new best macro AUC={best_macro_auc:.4f} -> saved {CHECKPOINT_PATH}")
if SMOKE_TEST:
    if not os.path.exists(CHECKPOINT_PATH):
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        _record_selection(CHECKPOINT_PATH, 1, "smoke-head", {"balanced_accuracy": float(bal_acc), "macro_auc": float(macro_auc)})
    print("SMOKE TEST: completed exactly one training batch and saved temporary checkpoint:", CHECKPOINT_PATH)


## 6. Phase 2 - fine-tune the last 4 transformer blocks (with early stopping)

Unfreeze `model.blocks[-4:]` + the final `norm` + the head, with attention-layer params at `base_lr * 0.1`. **Audit fix (1.2c):** the fixed 15-epoch schedule is replaced by up to `PHASE2_MAX_EPOCHS=40` epochs with early stopping (`PATIENCE=8`) on validation macro-AUC. The best checkpoint by macro-AUC is tracked across both phases.

In [ ]:
# Phase 2 — DECOUPLED two-stage training (Kang et al., 2020) with an optional
# logit-adjustment toggle (Menon et al., 2021), OR the original single-stage JOINT
# fine-tune. The branch is selected by config.TRAINING_MODE.
#
#   * decoupled: Stage A learns representations on the FULL network with
#     INSTANCE-balanced sampling (plain shuffled loader) + an UNWEIGHTED focal loss;
#     Stage B FREEZES the backbone and re-balances ONLY the classifier head with the
#     existing WEIGHTED sampler + UNWEIGHTED focal loss by default, then PROVES the freeze held.
#   * joint: the EXISTING last-4-blocks + final-norm + head unfreeze with two LR groups
#     (attn at BASE_LR*ATTN_LR_SCALE, others at BASE_LR), weighted sampler + weighted
#     focal; publication config guards against sampler+alpha double rebalancing.
from config import (TRAINING_MODE, STAGE_A_EPOCHS, STAGE_B_EPOCHS,
                    PHASE2_MAX_EPOCHS, PHASE2_PATIENCE,
                    USE_LOGIT_ADJUSTMENT, LOGIT_ADJUSTMENT_TAU, FOCAL_LOSS_GAMMA,
                    USE_WEIGHTED_SAMPLER_STAGE_B, USE_CLASS_WEIGHTED_LOSS_STAGE_B,
                    ALLOW_LEGACY_STAGE_B_DOUBLE_REBALANCING)
from training_utils import (class_priors_from_counts, LogitAdjustedLoss,
                            freeze_all_but_classifier, snapshot_frozen_params,
                            assert_frozen_unchanged, freeze_backbone_bn,
                            assert_single_stage_b_rebalancing,
                            save_resumable, load_resumable)

PATIENCE = PHASE2_PATIENCE   # early-stopping patience on val macro-AUC (config = single source).

# EMPIRICAL (un-rebalanced) class priors P(y=c)=count_c/total from the per-class TRAIN counts.
priors = class_priors_from_counts(class_sample_counts)


def _maybe_logit_adjust(base_criterion):
    """Optionally wrap a loss so tau*log(prior) is added to the logits first (Menon 2021)."""
    if USE_LOGIT_ADJUSTMENT:
        return LogitAdjustedLoss(base_criterion, priors, LOGIT_ADJUSTMENT_TAU).to(DEVICE)
    return base_criterion


def _run_stage(loader, criterion, optimizer, scheduler, max_epochs, patience,
               ckpt_path, tag, init_best=-1.0):
    """One early-stopping stage on val macro-AUC; saves this stage's best checkpoint.

    Reuses the notebook's OWN run_epoch(model, loader, criterion, optimizer) and
    evaluate(model, loader) -> (bal_acc, macro_auc, ...); appends to the shared `history`
    and keeps the global AMP `scaler` (inside run_epoch). Returns best val macro-AUC.
    """
    best_auc = init_best
    epochs_no_improve = 0
    start_epoch = 1
    resume_path = str(RESUME_DIR / ("agent_b_" + tag.lower() + "_resume.pth"))
    if SMOKE_TEST:
        print(f"[{tag}] skipped in smoke mode; head smoke batch already completed.")
        return init_best
    _r = load_resumable(resume_path, model, optimizer, scheduler, scaler, map_location=DEVICE)
    if _r is not None:
        start_epoch, best_auc, epochs_no_improve = _r["start_epoch"], _r["best_auc"], _r["epochs_no_improve"]
        print(f"[{tag}] RESUMED from {resume_path} -> start epoch {start_epoch} "
              f"(best={best_auc:.4f}, no_improve={epochs_no_improve})")
    for epoch in range(start_epoch, max_epochs + 1):
        train_loss = run_epoch(model, loader, criterion, optimizer)
        if scheduler is not None:
            scheduler.step()
        bal_acc, macro_auc, _, _ = evaluate(model, val_loader)
        history["loss"].append(train_loss)
        history["bal_acc"].append(bal_acc)
        history["macro_auc"].append(macro_auc)
        improved = ""
        if not math.isnan(macro_auc) and macro_auc > best_auc:
            best_auc = macro_auc
            epochs_no_improve = 0
            torch.save(model.state_dict(), ckpt_path)
            _record_selection(ckpt_path, epoch, tag, {"balanced_accuracy": float(bal_acc), "macro_auc": float(macro_auc)})
            improved = f"  -> saved best ({os.path.basename(ckpt_path)})"
        else:
            epochs_no_improve += 1
        print(f"[{tag} {epoch:02d}/{max_epochs}] loss={train_loss:.4f} "
              f"bal_acc={bal_acc:.4f} macro_auc={macro_auc:.4f} "
              f"no_improve={epochs_no_improve}{improved}")
        save_resumable(resume_path, tag, epoch, model, optimizer, scheduler, scaler, best_auc, epochs_no_improve)
        if epochs_no_improve >= patience:
            print(f"Early stopping {tag} at epoch {epoch} "
                  f"(no val macro-AUC gain for {patience} epochs). Best={best_auc:.4f}")
            break
    return best_auc


if TRAINING_MODE == "decoupled":
    # ====================== Stage A: representation learning ======================
    # UNFREEZE THE FULL NETWORK (every param trains).
    for p in model.parameters():
        p.requires_grad = True
    stageA_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    stageA_total = sum(p.numel() for p in model.parameters())
    print(f"[Stage A] full-network fine-tune: trainable {stageA_trainable:,} / {stageA_total:,}")

    # INSTANCE-balanced sampling: a PLAIN shuffled DataLoader over train_dataset with
    # NO WeightedRandomSampler (do NOT reuse the weighted train_loader here).
    stageA_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"), drop_last=True,
    )
    # UNWEIGHTED focal loss (alpha=None). Logit adjustment is applied at Stage B only.
    stageA_criterion = FocalLoss(gamma=FOCAL_LOSS_GAMMA, alpha=None).to(DEVICE)
    # Optimizer over ALL params at a LOW lr (~1e-5); cosine schedule over the Stage-A budget.
    stageA_optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=BASE_LR, weight_decay=WEIGHT_DECAY,
    )
    stageA_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        stageA_optimizer, T_max=STAGE_A_EPOCHS,
    )
    STAGEA_PATH = str(RESUME_DIR / "agent_b_stage_a_best.pth")
    stageA_best = _run_stage(stageA_loader, stageA_criterion, stageA_optimizer,
                             stageA_scheduler, STAGE_A_EPOCHS, PATIENCE,
                             STAGEA_PATH, tag="P2-A")
    # Reload the best Stage-A representation before re-balancing the classifier head.
    if not os.path.exists(STAGEA_PATH):
        torch.save(model.state_dict(), STAGEA_PATH)
    model.load_state_dict(torch.load(STAGEA_PATH, map_location=DEVICE))
    print(f"[Stage A] best val macro-AUC={stageA_best:.4f}; reloaded {STAGEA_PATH}")

    # ====================== Stage B: classifier re-balancing ======================
    n_head_trainable, n_total = freeze_all_but_classifier(model)
    n_bn_frozen = freeze_backbone_bn(model)  # cRT: also freeze backbone BN running stats (no-op for ViT/LayerNorm)
    print(f"[Stage B] froze {n_bn_frozen} BatchNorm module(s) so running stats are held too.")
    print(f"[Stage B] head-only re-balance: trainable {n_head_trainable:,} / {n_total:,}")
    snap = snapshot_frozen_params(model)   # capture BEFORE Stage B training.

    # Publication default: weighted sampler OR weighted focal loss, never both.
    assert_single_stage_b_rebalancing(USE_WEIGHTED_SAMPLER_STAGE_B,
                                      USE_CLASS_WEIGHTED_LOSS_STAGE_B,
                                      ALLOW_LEGACY_STAGE_B_DOUBLE_REBALANCING)
    stageB_loader = train_loader if USE_WEIGHTED_SAMPLER_STAGE_B else stageA_loader
    stageB_alpha = class_weights_tensor if USE_CLASS_WEIGHTED_LOSS_STAGE_B else None
    # (optionally logit-adjusted — the toggle is applied to the Stage-B loss only).
    stageB_criterion = _maybe_logit_adjust(
        FocalLoss(gamma=FOCAL_LOSS_GAMMA, alpha=stageB_alpha).to(DEVICE)
    )
    # Optimizer over ONLY the trainable (head) params at a HIGHER lr (~1e-3).
    stageB_optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR_HEAD, weight_decay=WEIGHT_DECAY,
    )
    stageB_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        stageB_optimizer, T_max=STAGE_B_EPOCHS,
    )
    # Save the FINAL best checkpoint to the nb's existing best path (agent_b_best.pth).
    best_macro_auc = _run_stage(stageB_loader, stageB_criterion, stageB_optimizer,
                                stageB_scheduler, STAGE_B_EPOCHS, PATIENCE,
                                CHECKPOINT_PATH, tag="P2-B")

    # PROVE the backbone never moved during Stage B (params bit-identical + still frozen).
    n_frozen = assert_frozen_unchanged(model, snap)
    print(f"[FREEZE VERIFIED] {n_frozen} backbone params bit-identical after Stage B.")
    print(f"Best Stage-B (final) val macro-AUC: {best_macro_auc:.4f} -> {CHECKPOINT_PATH}")

else:  # "joint" — EXISTING single-stage Phase-2 fine-tune; behaviour UNCHANGED.
    # AUDIT (1.1): the previous version ran a FIXED 15 epochs with no early stopping.
    # AUDIT (1.2c) fix: extend the budget and add early stopping on validation macro-AUC.
    # Publication default: unweighted focal loss; Stage-B rebalancing is controlled by config.
    # The unfreeze depth (last 4 blocks) is intentionally left unchanged.
    for p in model.parameters():
        p.requires_grad = False
    for blk in list(model.blocks)[-4:]:
        for p in blk.parameters():
            p.requires_grad = True
    if hasattr(model, "norm") and isinstance(model.norm, nn.Module):
        for p in model.norm.parameters():
            p.requires_grad = True
    for p in model.get_classifier().parameters():
        p.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable params (last 4 blocks + norm + head): {trainable:,} ({trainable/1e6:.2f}M)")
    print("  AUDIT 1.2d: well above the 10M 'insufficient capacity' threshold; depth left unchanged.")

    # Two param groups: attention-layer params ('.attn.') at base_lr*0.1, others at base_lr.
    attn_params, other_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if ".attn." in name:
            attn_params.append(p)
        else:
            other_params.append(p)
    print(f"attn params: {sum(p.numel() for p in attn_params):,} | "
          f"other params: {sum(p.numel() for p in other_params):,}")

    # Weighted focal loss = the existing global `criterion`, optionally logit-adjusted.
    joint_criterion = _maybe_logit_adjust(criterion)

    finetune_optimizer = torch.optim.AdamW(
        [
            {"params": attn_params, "lr": BASE_LR * ATTN_LR_SCALE},
            {"params": other_params, "lr": BASE_LR},
        ],
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        finetune_optimizer, T_max=PHASE2_MAX_EPOCHS,
    )
    # Continue the cross-phase best tracking from Phase 1 (init_best=best_macro_auc), saving
    # the global best to CHECKPOINT_PATH — identical to the original single-stage loop.
    best_macro_auc = _run_stage(train_loader, joint_criterion, finetune_optimizer,
                                scheduler, PHASE2_MAX_EPOCHS, PATIENCE,
                                CHECKPOINT_PATH, tag="P2", init_best=best_macro_auc)
    print(f"Best macro AUC across both phases: {best_macro_auc:.4f}")


## 7. Training curves

Plot training loss, validation balanced accuracy and validation macro AUC across the
combined Phase 1 + Phase 2 schedule. The dashed line marks the P1 -> P2 boundary.

In [ ]:
epochs_axis = np.arange(1, len(history["loss"]) + 1)
phase_boundary = MAX_EPOCHS_HEAD + 0.5

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
panels = [
    ("loss", "Training loss", "tab:red"),
    ("bal_acc", "Val balanced accuracy", "tab:blue"),
    ("macro_auc", "Val macro AUC", "tab:green"),
]
for ax, (key, title, color) in zip(axes, panels):
    ax.plot(epochs_axis, history[key], marker="o", color=color)
    ax.axvline(phase_boundary, linestyle="--", color="gray", alpha=0.7, label="P1->P2")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(title)
    ax.legend()
plt.suptitle("Agent B (ViT-B/16) training curves", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, "figures", "agent_b_curves.png"), dpi=120, bbox_inches="tight")
plt.show()

## 8. Reload best checkpoint

Reload the best weights observed during training (already serialized to
`artifacts/rescue/checkpoints/agent_b_best.pth` whenever `model_val` macro AUC improved) for the downstream
confusion matrix and attention-rollout visualizations.

In [ ]:
best_state = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(best_state)
model.eval()
print(f"Reloaded best Agent B checkpoint from {CHECKPOINT_PATH}")
print(f"File size: {os.path.getsize(CHECKPOINT_PATH) / 1e6:.1f} MB")

bal_acc, macro_auc, _, _ = evaluate(model, val_loader)
_write_run_metadata(
    agent_id="agent_b",
    checkpoint_path=CHECKPOINT_PATH,
    architecture=MODEL_NAME,
    pretrained_source=PRETRAINED_SOURCE,
    optimizer_config={"name": "AdamW", "head_lr": LR_HEAD, "representation_lr": BASE_LR, "attention_lr_scale": ATTN_LR_SCALE, "weight_decay": WEIGHT_DECAY},
    scheduler_config={"name": "CosineAnnealingLR", "stage_a_t_max": STAGE_A_EPOCHS, "stage_b_t_max": STAGE_B_EPOCHS},
    loss_config={"name": "FocalLoss", "gamma": float(FOCAL_LOSS_GAMMA), "stage_a_alpha": None, "stage_b_class_weighted": bool(USE_CLASS_WEIGHTED_LOSS_STAGE_B)},
    sampling_config={"stage_a": "ordinary shuffled DataLoader", "stage_b": "WeightedRandomSampler" if USE_WEIGHTED_SAMPLER_STAGE_B else "ordinary shuffled DataLoader"},
    validation_metrics={"balanced_accuracy": float(bal_acc), "macro_auc": float(macro_auc)},
    transform_config={"train": repr(build_train_transforms()), "model_val": repr(build_val_transforms()), "image_size": IMAGE_SIZE, "mean": IMAGENET_MEAN, "std": IMAGENET_STD},
)


## 9. Confusion matrix

Run the best model over the validation split and render a row-normalized per-class
confusion-matrix heatmap.

In [ ]:
from sklearn.metrics import confusion_matrix

_, _, y_true, y_pred = evaluate(model, val_loader)
cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
row_sums = cm.sum(axis=1, keepdims=True)
cm_norm = np.divide(cm, np.clip(row_sums, 1, None), dtype=np.float64)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="cividis",
            xticklabels=ISIC_CLASSES, yticklabels=ISIC_CLASSES, ax=ax,
            cbar_kws={"label": "Row-normalized frequency"})
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title("Agent B - validation confusion matrix (row-normalized)")
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, "figures", "agent_b_confusion.png"), dpi=120, bbox_inches="tight")
plt.show()

## 9b. Per-class validation report (audit item 1.3)
Explicit balanced accuracy and per-class recall on the held-out split, flagging any class whose recall is below 0.40, so minority-class behaviour is visible immediately after a Kaggle run. Reuses `y_true`/`y_pred` from the confusion-matrix cell above.

In [ ]:
from sklearn.metrics import classification_report, balanced_accuracy_score, recall_score

bal_acc = balanced_accuracy_score(y_true, y_pred)
print('Final validation balanced accuracy: %.4f' % bal_acc)
print('\nPer-class report:')
print(classification_report(y_true, y_pred, labels=list(range(NUM_CLASSES)),
                            target_names=ISIC_CLASSES, digits=4, zero_division=0))
print('\nFlag: any class recall below 0.40?')
recalls = recall_score(y_true, y_pred, average=None, labels=list(range(NUM_CLASSES)), zero_division=0)
for cls, rec in zip(ISIC_CLASSES, recalls):
    flag = '  <-- LOW' if rec < 0.40 else ''
    print('  %-4s recall=%.4f%s' % (cls, rec, flag))


## 10. Attention rollout visualization

This optional visualization is disabled by default during publication training; set `ARGUS_RUN_VISUALIZATIONS=1` to enable it. ViTs do not have a conv feature map, so instead of Grad-CAM we use **attention rollout**
(Abnar & Zuidema, 2020). We disable fused attention, register a forward hook on every
`block.attn` to capture its input, recompute the per-head softmax attention from the
block's `qkv`, average heads, form `A_hat = 0.5*(A + I)`, row-normalize, and multiply
across all layers. The CLS row over the 196 patch tokens is reshaped to 14x14, upsampled
to 224x224 and overlaid on a 3x3 grid of validation images. Hooks are removed in a
`finally` block.

In [ ]:
if SMOKE_TEST or not RUN_VISUALIZATIONS:
    print("Optional attention rollout generation skipped (set ARGUS_RUN_VISUALIZATIONS=1 to enable).")
else:
    import cv2

    model.eval()
    mean = np.array(IMAGENET_MEAN, dtype=np.float32)
    std = np.array(IMAGENET_STD, dtype=np.float32)

    # Disable timm fused attention so we can recompute per-head softmax weights ourselves.
    for blk in model.blocks:
        if hasattr(blk.attn, "fused_attn"):
            blk.attn.fused_attn = False

    # Geometry: ViT-B/16 at 224 => 14x14 = 196 patch tokens + 1 CLS = 197 tokens.
    GRID = IMAGE_SIZE // 16  # 14
    NUM_PATCH_TOKENS = GRID * GRID

    captured = []  # one captured (N, T, T) attention matrix per block, in forward order


    def make_hook(attn_module):
        def hook(module, inputs, output):
            x = inputs[0]                      # (N, T, C) input to the attention block
            N, T, C = x.shape
            num_heads = int(module.num_heads)
            head_dim = C // num_heads
            scale = getattr(module, "scale", None)
            if scale is None:
                scale = head_dim ** -0.5
            qkv = module.qkv(x).reshape(N, T, 3, num_heads, head_dim)
            qkv = qkv.permute(2, 0, 3, 1, 4)   # (3, N, heads, T, head_dim)
            q, k = qkv[0], qkv[1]
            attn = (q @ k.transpose(-2, -1)) * float(scale)
            attn = attn.softmax(dim=-1)        # (N, heads, T, T)
            captured.append(attn.mean(dim=1).detach().to(torch.float32).cpu())  # avg heads
        return hook


    def attention_rollout(input_tensor):
        """Return (224x224 heatmap in [0,1], predicted_class_index) for one image tensor."""
        captured.clear()
        with torch.no_grad():
            logits = model(input_tensor)
            pred_idx = int(torch.softmax(logits, dim=1).argmax(dim=1).item())
        # Multiply A_hat = 0.5*(A + I), row-normalized, across all layers.
        T = captured[0].shape[-1]
        eye = torch.eye(T, dtype=torch.float32)
        result = eye.clone()
        for attn in captured:
            a_hat = 0.5 * attn[0] + 0.5 * eye
            a_hat = a_hat / a_hat.sum(dim=-1, keepdim=True).clamp_min(1e-12)
            result = a_hat @ result
        # CLS row (token 0) over the 196 patch tokens (drop the CLS self-attention column).
        cls_attn = result[0, 1:][:NUM_PATCH_TOKENS]
        grid = cls_attn.reshape(GRID, GRID).numpy().astype(np.float32)
        heat = cv2.resize(grid, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_CUBIC).astype(np.float32)
        span = float(heat.max() - heat.min())
        if span < 1e-12:
            heat = np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32)
        else:
            heat = (heat - heat.min()) / span
        return np.clip(heat, 0.0, 1.0).astype(np.float32), pred_idx


    grid_rows = model_val_df.sample(n=9, random_state=SEED).reset_index(drop=True)
    handles = []
    try:
        for blk in model.blocks:
            handles.append(blk.attn.register_forward_hook(make_hook(blk.attn)))

        fig, axes = plt.subplots(3, 3, figsize=(12, 12))
        for ax, (_, row) in zip(axes.ravel(), grid_rows.iterrows()):
            pil = Image.open(row["path"]).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))
            rgb = np.asarray(pil, dtype=np.float32) / 255.0
            norm = (rgb - mean) / std
            input_tensor = torch.from_numpy(norm.transpose(2, 0, 1)).unsqueeze(0).to(DEVICE)
            heat, pred_idx = attention_rollout(input_tensor)
            heatmap = cv2.applyColorMap((heat * 255).astype(np.uint8), cv2.COLORMAP_JET)
            heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
            overlay = np.clip(0.55 * rgb + 0.45 * heatmap, 0.0, 1.0)
            ax.imshow(overlay)
            true_idx = int(row["label"])
            ax.set_title(f"true={ISIC_CLASSES[true_idx]} / pred={ISIC_CLASSES[pred_idx]}",
                         fontsize=10, color=("green" if pred_idx == true_idx else "red"))
            ax.axis("off")
        plt.suptitle("Agent B - attention rollout (CLS -> patch)", fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(WORK_DIR, "figures", "agent_b_rollout.png"), dpi=120, bbox_inches="tight")
        plt.show()
    finally:
        for h in handles:
            h.remove()
        print("Removed all attention hooks.")


## Publication outputs

The full run writes `artifacts/rescue/checkpoints/agent_b_best.pth`, resumable stage state under `artifacts/rescue/checkpoints/resume/`, and `artifacts/rescue/checkpoints/agent_b_run_metadata.json`.
